# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset Name: {}".format(metadata_json.get('name')))
print("Dataset Description: {}".format(metadata_json.get('description')))


## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

Below we summarize the dataset structure using the `@id` for all entities, which provides unambiguous references for further data extraction and analysis.

In [ ]:
# List available record sets (by @id), their fields, and columns
record_sets = metadata_json.get('recordSet', [])
if not record_sets:
    print('No record sets found in metadata. Attempting to extract from dataset itself...')
    # Optionally, dataset.record_sets if mlcroissant exposes this
    record_sets = dataset.record_sets

recordset_ids = []
for rs in record_sets:
    # Each record set can be just an @id string or a dict with @id
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    print(f'RecordSet @id: {rs_id}')
    recordset_ids.append(rs_id)
    # Optionally, print details about fields and columns
    try:
        recordset_obj = dataset.metadata.get_by_id(rs_id)
        fields = recordset_obj.get('field', []) if hasattr(recordset_obj, 'get') else []
        print(f'  Fields in RecordSet:')
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f'    Field @id: {field_id}')
            # Print columns if available
            if isinstance(field, dict) and 'column' in field:
                columns = field['column']
                for col in columns:
                    col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
                    print(f'      Column @id: {col_id}')
    except Exception as e:
        # In some cases, the metadata structure may not allow fetching fields or columns directly
        print(f'  Unable to inspect fields for RecordSet {rs_id}: {str(e)}')
print('\nAll RecordSet @id values:', recordset_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are via `@id` fields.

If multiple record sets exist, extract all, otherwise extract the main survey record set.

In [ ]:
# Extract data from each record set
# For this dataset, we'll assume a single main record set if none are listed
if not recordset_ids:
    # In some schemas, recordSet may not be explicitly listed in metadata, but mlcroissant exposes available IDs
    recordset_ids = dataset.record_sets  # fallback

dataframes = {}
for recordset_id in recordset_ids:
    records = list(dataset.records(record_set=recordset_id))
    df = pd.DataFrame(records)
    dataframes[recordset_id] = df
    print(f"\nColumns for RecordSet {recordset_id}: {df.columns.tolist()}")
    print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using fields referenced by their `@id`. We'll demonstrate filtering, normalization, grouping, and basic summary statistics.

In [ ]:
# Choose a RecordSet for analysis
if recordset_ids:
    record_set_id = recordset_ids[0]

    df = dataframes[record_set_id]
    print(f"Analyzing RecordSet {record_set_id}.")
    # Let's find numeric columns (scores, ages, etc)
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric columns in RecordSet {record_set_id}: {numeric_cols}")

    # Example: use GAD-7 Score, PHQ-9 Score, PSQ Score or Age
    # We'll attempt with 'gad7_score', 'phq9_score', 'psq_score', 'age' -- get their IDs
    prefer_fields = ['gad7_score', 'phq9_score', 'psq_score', 'age']
    numeric_field = None
    for nf in prefer_fields:
        if nf in df.columns:
            numeric_field = nf
            break
    if numeric_field is None:
        # Use the very first numeric column
        numeric_field = numeric_cols[0] if numeric_cols else df.columns[0]

    print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

    # Filter records above a threshold
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Grouping by a categorical field
    group_fields = ['gender', 'village', 'level_of_education', 'marital_status']
    group_field = None
    for gf in group_fields:
        if gf in df.columns:
            group_field = gf
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize distributions of key fields, e.g., GAD-7, PHQ-9, or PSQ scores, grouped by demographic fields, using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if recordset_ids:
    record_set_id = recordset_ids[0]
    df = dataframes[record_set_id]

    # Histogram of GAD-7 or PHQ-9 scores
    score_fields = ['gad7_score', 'phq9_score', 'psq_score']
    score_field = None
    for sf in score_fields:
        if sf in df.columns:
            score_field = sf
            break
    if score_field:
        plt.figure(figsize=(8,6))
        sns.histplot(df[score_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {score_field} (RecordSet @id: {record_set_id})")
        plt.xlabel(score_field)
        plt.ylabel("Count")
        plt.show()

        # Boxplot by gender or education
        category_fields = ['gender', 'level_of_education', 'marital_status', 'village']
        for cat_field in category_fields:
            if cat_field in df.columns:
                plt.figure(figsize=(10,5))
                sns.boxplot(x=cat_field, y=score_field, data=df)
                plt.title(f"{score_field} by {cat_field} (RecordSet @id: {record_set_id})")
                plt.xlabel(cat_field)
                plt.ylabel(score_field)
                plt.show()
                break
    else:
        print("No GAD-7/PHQ-9/PSQ score fields available for visualization.")
else:
    print("Unable to visualize: no record sets loaded.")

## 6. Conclusion
This notebook provided an end-to-end demonstration of exploring a Croissant-compliant dataset using `mlcroissant`, referencing all entities by their `@id`.
We loaded metadata, inspected data structure, extracted and analyzed survey records, performed filtering and normalization, and visualized key metrics.

Key findings, such as elevated GAD-7 or PHQ-9 scores in certain demographic groups, can now be further investigated.

For reproducibility, ensure all fields and record sets are referenced by their `@id` throughout. Explore more advanced analysis by integrating other ML and statistical tools.